# Autoresearch Experiment Analysis (x-transformers-rl)

Analysis of autonomous RL hyperparameter tuning results from `results.tsv`.
Metric: **mean_reward** on CartPole-v1 — higher is better. Solved = 475+.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 5 columns: commit, mean_reward, memory_gb, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["mean_reward"] = pd.to_numeric(df["mean_reward"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    reward = row["mean_reward"]
    desc = row["description"]
    print(f"  #{i:3d}  reward={reward:.4f}  mem={row['memory_gb']:.1f}GB  {desc}")

## Mean Reward Over Time

Track how the best (kept) mean_reward evolves as experiments progress.
The running maximum shows the "frontier" — the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

baseline_reward = valid.loc[0, "mean_reward"]

# Compute running max from kept experiments
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_reward = valid.loc[kept_mask, "mean_reward"]
running_max = kept_reward.cummax()
best = running_max.iloc[-1] if len(running_max) > 0 else baseline_reward

# Plot discarded as faint background dots
disc = valid[valid["status"] == "DISCARD"]
ax.scatter(
    disc.index,
    disc["mean_reward"],
    c="#cccccc",
    s=12,
    alpha=0.5,
    zorder=2,
    label="Discarded",
)

# Plot kept experiments as prominent green dots
kept_v = valid[valid["status"] == "KEEP"]
ax.scatter(
    kept_v.index,
    kept_v["mean_reward"],
    c="#2ecc71",
    s=50,
    zorder=4,
    label="Kept",
    edgecolors="black",
    linewidths=0.5,
)

# Running maximum step line
ax.step(
    kept_idx,
    running_max,
    where="post",
    color="#27ae60",
    linewidth=2,
    alpha=0.7,
    zorder=3,
    label="Running best",
)

# Solved threshold line
ax.axhline(y=475, color="red", linestyle="--", alpha=0.5, label="Solved (475)")

# Label each kept experiment with its description
for idx, reward in zip(kept_idx, kept_reward):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."

    ax.annotate(
        desc,
        (idx, reward),
        textcoords="offset points",
        xytext=(6, 6),
        fontsize=8.0,
        color="#1a7a3a",
        alpha=0.9,
        rotation=30,
        ha="left",
        va="bottom",
    )

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Mean Reward (higher is better)", fontsize=12)
ax.set_title(
    f"Autoresearch (x-transformers-rl) Progress: {n_total} Experiments, {n_kept} Kept Improvements",
    fontsize=14,
)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)

# Y-axis: from below baseline to above best
margin = max((best - baseline_reward) * 0.15, 10)
ax.set_ylim(max(0, baseline_reward - margin), best + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
# Summary stats
kept = df[df["status"] == "KEEP"].copy()
baseline_reward = df.iloc[0]["mean_reward"]
best_reward = kept["mean_reward"].max()
best_row = kept.loc[kept["mean_reward"].idxmax()]

print(f"Baseline mean_reward:  {baseline_reward:.4f}")
print(f"Best mean_reward:      {best_reward:.4f}")
print(
    f"Total improvement:     {best_reward - baseline_reward:+.4f} ({(best_reward - baseline_reward) / baseline_reward * 100:+.1f}%)"
)
print(f"Best experiment:       {best_row['description']}")
print(f"Solved (475+):         {'YES' if best_reward >= 475 else 'NO'}")
print()

# How many experiments to find each improvement
print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: reward={row['mean_reward']:.4f}  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta is measured vs the previous kept experiment's reward
# (since experiments are cumulative — each one builds on the last kept state)
kept = df[df["status"] == "KEEP"].copy()
kept["prev_reward"] = kept["mean_reward"].shift(1)
kept["delta"] = kept["mean_reward"] - kept["prev_reward"]

# Drop baseline (no delta)
hits = kept.iloc[1:].copy()

# Sort by delta improvement (biggest first)
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'Reward':>10}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(
        f"{rank:4d}  {row['delta']:+.4f}  {row['mean_reward']:.4f}  {row['description']}"
    )

print(
    f"\n{'':>4}  {hits['delta'].sum():+.4f}  {'':>10}  TOTAL improvement over baseline"
)

## Memory Usage

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)
colors = ["#2ecc71" if s == "KEEP" else "#cccccc" for s in valid["status"]]
ax.bar(valid.index, valid["memory_gb"], color=colors, alpha=0.7, width=0.8)
ax.axhline(y=24.0, color="red", linestyle="--", alpha=0.5, label="VRAM limit (24 GB)")
ax.set_xlabel("Experiment #", fontsize=11)
ax.set_ylabel("Peak Memory (GB)", fontsize=11)
ax.set_title("Peak VRAM Usage Per Experiment", fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2, axis="y")

plt.tight_layout()
plt.savefig("memory.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to memory.png")